# Fundamentals curiosity — World Oil Supply & Global Oil Demand

**Not a freight input.** Platts share views in Snowflake for browsing only. Freight physical tape stays in notebooks 01/02 (ICE + Kpler).

Caveats (verified 2026-10-07):
- Views last altered **2026-07-21** — not tick-live.
- DEMAND: `History` ends **2024-12**; **2025+ is all `Forecast`**.
- PRODUCTION: no row-level as-of; use `Short Term` (monthly) vs `Long Term` (`MONTH=0`).
- Units are **MBD** (thousand b/d). Not VLCC liftings or exports.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT.parent / "desk_core"))

from desk_core.snowflake.connector import SnowflakeConfig, SnowflakeConnector

pd.options.display.float_format = lambda x: f"{x:,.2f}"
TAG = "freight-agent:fundamentals-curiosity"
CACHE = ROOT / "data" / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
REFRESH = False
print("ROOT", ROOT)

ROOT C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\freight_dashboard


In [2]:
def cached_parquet(name, fetch):
    path = CACHE / f"{name}.parquet"
    if path.exists() and not REFRESH:
        return pd.read_parquet(path)
    df = fetch()
    df.to_parquet(path, index=False)
    return df


def sf_query(sql: str) -> pd.DataFrame:
    cfg = SnowflakeConfig(
        database="SPGICI_WORLDOILSUPPLY",
        schema="SUPPLY",
        role="JG_GENERAL",
        warehouse="EU_COMMOD_RESEARCH_XS_WH",
    )
    with SnowflakeConnector(cfg) as conn:
        cur = conn.cursor()
        cur.execute(f"ALTER SESSION SET QUERY_TAG = '{TAG}'")
        cur.execute(sql)
        return pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])


def ym_index(df):
    out = df.copy()
    out["Date"] = pd.to_datetime(
        dict(year=out["YEAR"], month=out["MONTH"].clip(lower=1), day=1)
    )
    return out

## Freshness snapshot

Object `LAST_ALTERED` vs what the rows claim (History vs Forecast).

In [3]:
meta = cached_parquet(
    "fund_curiosity_object_meta",
    lambda: sf_query(
        """
        SELECT TABLE_CATALOG, TABLE_SCHEMA, TABLE_NAME, TABLE_TYPE, CREATED, LAST_ALTERED
        FROM SPGICI_WORLDOILSUPPLY.INFORMATION_SCHEMA.TABLES
        WHERE TABLE_SCHEMA = 'SUPPLY' AND TABLE_NAME = 'PRODUCTION'
        UNION ALL
        SELECT TABLE_CATALOG, TABLE_SCHEMA, TABLE_NAME, TABLE_TYPE, CREATED, LAST_ALTERED
        FROM SPGICI_GLOBALOILDEMAND.INFORMATION_SCHEMA.TABLES
        WHERE TABLE_SCHEMA = 'MGODF' AND TABLE_NAME = 'DEMAND'
        """
    ),
)
dq = cached_parquet(
    "fund_curiosity_demand_quality",
    lambda: sf_query(
        """
        SELECT DATA_QUALITY_NAME, MIN(YEAR) YMIN, MAX(YEAR) YMAX, COUNT(*) N
        FROM SPGICI_GLOBALOILDEMAND.MGODF.DEMAND
        GROUP BY 1 ORDER BY 1
        """
    ),
)
display(meta)
display(dq)
display(Markdown(
    "If `LAST_ALTERED` is weeks old and DEMAND `History` max year is still 2024, "
    "treat 2025–26 demand as a **forecast case**, not actuals."
))

,TABLE_CATALOG,TABLE_SCHEMA,TABLE_NAME,TABLE_TYPE,CREATED,LAST_ALTERED
0,SPGICI_WORLDOILSUPPLY,SUPPLY,PRODUCTION,VIEW,2023-11-08 20:29:39.539000+00:00,2026-07-21 16:04:17.801000+00:00
1,SPGICI_GLOBALOILDEMAND,MGODF,DEMAND,VIEW,2024-05-07 17:09:25.807000+00:00,2026-07-21 16:04:16.176000+00:00


,DATA_QUALITY_NAME,YMIN,YMAX,N
0,Forecast,2025,2028,72960
1,History,2015,2024,182400


If `LAST_ALTERED` is weeks old and DEMAND `History` max year is still 2024, treat 2025–26 demand as a **forecast case**, not actuals.

## Saudi Arabia — Short Term crude production (MBD)

Sum of grade-level `Production` rows. 2026 step-down is the share’s scenario path (Hormuz-era), not a live export print.

In [4]:
saudi = cached_parquet(
    "fund_curiosity_saudi_crude_st",
    lambda: sf_query(
        """
        SELECT YEAR, MONTH, GRADE_NAME, VALUE
        FROM SPGICI_WORLDOILSUPPLY.SUPPLY.PRODUCTION
        WHERE SCENARIO_TERM_NAME = 'Short Term'
          AND COUNTRY_NAME = 'Saudi Arabia'
          AND SUPPLY_TYPE_NAME = 'Crude'
          AND PRODUCTION_TYPE_NAME = 'Production'
          AND YEAR BETWEEN 2022 AND 2027
          AND MONTH BETWEEN 1 AND 12
        """
    ),
)
saudi = ym_index(saudi)
saudi_tot = (
    saudi.groupby("Date", as_index=False)["VALUE"]
    .sum()
    .rename(columns={"VALUE": "crude_mbd"})
)
saudi_grade = saudi.pivot_table(index="Date", columns="GRADE_NAME", values="VALUE", aggfunc="sum")

fig = go.Figure()
fig.add_trace(go.Scatter(x=saudi_tot["Date"], y=saudi_tot["crude_mbd"], name="Total crude", line=dict(width=3)))
for col in saudi_grade.columns:
    fig.add_trace(go.Scatter(x=saudi_grade.index, y=saudi_grade[col], name=str(col), visible="legendonly"))
fig.update_layout(
    title="Saudi Arabia — Short Term crude production (MBD)",
    template="plotly_white", height=420, legend=dict(orientation="h"),
    yaxis_title="MBD",
)
fig.show()
display(saudi_tot.tail(6))

,Date,crude_mbd
66,2027-07-01,"8,025.00"
67,2027-08-01,"8,125.00"
68,2027-09-01,"8,125.00"
69,2027-10-01,"8,324.99"
70,2027-11-01,"8,325.00"
71,2027-12-01,"8,324.99"


## MEG producers — Short Term crude (stacked)

Saudi, UAE, Iraq, Kuwait, Iran — monthly total crude production.

In [5]:
meg = cached_parquet(
    "fund_curiosity_meg_crude_st",
    lambda: sf_query(
        """
        SELECT YEAR, MONTH, COUNTRY_NAME, SUM(VALUE) AS CRUDE_MBD
        FROM SPGICI_WORLDOILSUPPLY.SUPPLY.PRODUCTION
        WHERE SCENARIO_TERM_NAME = 'Short Term'
          AND SUPPLY_TYPE_NAME = 'Crude'
          AND PRODUCTION_TYPE_NAME = 'Production'
          AND YEAR BETWEEN 2022 AND 2027
          AND MONTH BETWEEN 1 AND 12
          AND COUNTRY_NAME IN (
            'Saudi Arabia', 'United Arab Emirates', 'Iraq', 'Kuwait', 'Iran'
          )
        GROUP BY 1, 2, 3
        """
    ),
)
meg = ym_index(meg)
meg_w = meg.pivot_table(index="Date", columns="COUNTRY_NAME", values="CRUDE_MBD", aggfunc="sum")

fig = go.Figure()
for col in meg_w.columns:
    fig.add_trace(go.Scatter(x=meg_w.index, y=meg_w[col], name=str(col), stackgroup="one"))
fig.update_layout(
    title="MEG Short Term crude production (stacked MBD)",
    template="plotly_white", height=420, legend=dict(orientation="h"),
    yaxis_title="MBD",
)
fig.show()

## China — product demand (MBD)

Solid line = `History`, dotted = `Forecast`. Expect the colour change at 2025.

In [6]:
chn = cached_parquet(
    "fund_curiosity_china_demand",
    lambda: sf_query(
        """
        SELECT YEAR, MONTH, PRODUCT_NAME, VALUE, DATA_QUALITY_NAME
        FROM SPGICI_GLOBALOILDEMAND.MGODF.DEMAND
        WHERE COUNTRY_NAME ILIKE '%China%'
          AND YEAR BETWEEN 2018 AND 2027
          AND PRODUCT_NAME IN (
            'Gasoil/Diesel',
            'Motor Gasoline: Fin. And Blen. Comp.',
            'Jet Fuel Kerosene',
            'Naphtha',
            'Crude Burn'
          )
        """
    ),
)
chn = ym_index(chn)

fig = go.Figure()
for prod, g in chn.groupby("PRODUCT_NAME"):
    hist = g.loc[g["DATA_QUALITY_NAME"] == "History"]
    fc = g.loc[g["DATA_QUALITY_NAME"] == "Forecast"]
    if not hist.empty:
        fig.add_trace(go.Scatter(x=hist["Date"], y=hist["VALUE"], name=f"{prod} (hist)"))
    if not fc.empty:
        fig.add_trace(
            go.Scatter(
                x=fc["Date"], y=fc["VALUE"], name=f"{prod} (fcst)",
                line=dict(dash="dot"),
            )
        )
fig.update_layout(
    title="China product demand — History vs Forecast (MBD)",
    template="plotly_white", height=480, legend=dict(orientation="h"),
    yaxis_title="MBD",
)
fig.show()

tot = chn.groupby(["Date", "DATA_QUALITY_NAME"], as_index=False)["VALUE"].sum()
fig2 = go.Figure()
for q, g in tot.groupby("DATA_QUALITY_NAME"):
    fig2.add_trace(
        go.Scatter(
            x=g["Date"], y=g["VALUE"], name=str(q),
            line=dict(dash="dot" if q == "Forecast" else "solid", width=3),
        )
    )
fig2.update_layout(
    title="China — sum of selected products (MBD)",
    template="plotly_white", height=360, legend=dict(orientation="h"),
)
fig2.show()

## Side-by-side curiosity: Saudi crude vs China gasoil

Different concepts (producer crude vs product demand). Only for visual scale — no freight claim.

In [7]:
gas = chn.loc[chn["PRODUCT_NAME"] == "Gasoil/Diesel", ["Date", "VALUE", "DATA_QUALITY_NAME"]].rename(
    columns={"VALUE": "china_gasoil_mbd"}
)
merged = saudi_tot.merge(gas, on="Date", how="inner")

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Scatter(x=merged["Date"], y=merged["crude_mbd"], name="Saudi crude ST"),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=merged["Date"], y=merged["china_gasoil_mbd"], name="China gasoil",
        line=dict(dash="dot"),
    ),
    secondary_y=True,
)
fig.update_layout(
    title="Curiosity overlay — Saudi crude vs China gasoil (MBD)",
    template="plotly_white", height=400, legend=dict(orientation="h"),
)
fig.update_yaxes(title_text="Saudi crude MBD", secondary_y=False)
fig.update_yaxes(title_text="China gasoil MBD", secondary_y=True)
fig.show()
display(Markdown(
    f"Overlap months: **{len(merged)}**. "
    f"China series quality in overlap: {merged['DATA_QUALITY_NAME'].value_counts().to_dict()}."
))

Overlap months: **72**. China series quality in overlap: {'History': 36, 'Forecast': 36}.

## Takeaway

- Nice for **balance-sheet curiosity** and seeing how Platts labelled History vs Forecast.
- **Do not** wire into freight tools A–H; Kpler + ICE remain the tape.
- Flip `REFRESH = True` after a share refresh if `LAST_ALTERED` moves.